# 06 - Business Recommendations and Operational Insights

This notebook translates the modelling results into concrete business recommendations.
The goal is to demonstrate how predictive analytics can improve decision making for
retail operations, particularly in demand planning and stock optimisation.

## Why this step matters

Technical accuracy alone does not solve business problems. Stakeholders want to know:
- What actions should be taken.
- Which stores need attention.
- How forecasting improvements translate into financial or operational value.
- Where the model can be safely deployed.
- What risks remain.

This notebook consolidates findings into insights that directly inform planning, 
inventory management, budgeting, and operational strategy.


## Summary of Key Forecasting KPIs

Before giving recommendations, it is essential to summarise the model's performance
in simple, digestible terms. This provides context and supports the justification
for using the model in operational workflows.

A clear KPI summary helps:
- Communicate improvements over the baseline.
- Quantify the value of the model.
- Provide management-ready talking points.


In [1]:
import pandas as pd

# Create a concise KPI comparison table manually or by loading from earlier outputs

kpi_data = {
    "model": ["Baseline (lag_1)", "Random Forest"],
    "mae": [50730.91, 36792.57],
    "rmse": [75702.75, 55665.33],
    "mape": [4.94, 3.52],
}

kpi_df = pd.DataFrame(kpi_data)

# Display KPI summary
kpi_df


,model,mae,rmse,mape
0,Baseline (lag_1),50730.91,75702.75,4.94
1,Random Forest,36792.57,55665.33,3.52


## Quantifying Model Improvement

Business stakeholders need to understand the size of the improvement, not just see
the numbers. Converting MAE and MAPE reductions into percentage improvements makes
the benefits immediately clear.

This strengthens the case for model adoption and demonstrates tangible impact on
inventory efficiency and forecasting reliability.


In [3]:
# Calculate improvements relative to baseline

mae_improvement = (1 - kpi_df.loc[1, "mae"] / kpi_df.loc[0, "mae"]) * 100
rmse_improvement = (1 - kpi_df.loc[1, "rmse"] / kpi_df.loc[0, "rmse"]) * 100
mape_improvement = (1 - kpi_df.loc[1, "mape"] / kpi_df.loc[0, "mape"]) * 100

print(f"MAE improvement: {mae_improvement:.1f}%")
print(f"RMSE improvement: {rmse_improvement:.1f}%")
print(f"MAPE improvement: {mape_improvement:.1f}%b")


MAE improvement: 27.5%
RMSE improvement: 26.5%
MAPE improvement: 28.7%


## Identifying Stores Requiring Operational Attention

Not all stores perform equally. Some exhibit higher forecast error and persistent bias,
which may lead to operational inefficiencies.

**Why this matters:**
- High-error stores face elevated risk of stockouts or overstocking.
- These stores may require local adjustments, additional monitoring, or supplemental features.
- Prioritising attention ensures effective resource allocation.

We analyse:
- The top 10 stores by MAPE (already computed).
- Stores with significant positive bias (systematic over forecasting).


## Define project paths

Notebook 6 runs independently, so we must redefine the project directory structure
used in previous notebooks. This ensures consistency when loading processed data
and evaluation outputs.


In [6]:
# Recreate directory structure used in earlier notebooks

from pathlib import Path

# Identify project root (one level above the current notebook folder)
# Adjust as needed depending on your folder structure.
PROJECT_ROOT = Path.cwd().resolve().parent

# Define key data directories
DATA_DIR = PROJECT_ROOT / "data"
DATA_PROCESSED = DATA_DIR / "processed"

# Evaluation outputs directory
eval_path = DATA_PROCESSED / "evaluation"

# Confirm paths exist
print("Project root:", PROJECT_ROOT)
print("Processed data path:", DATA_PROCESSED)
print("Evaluation data path:", eval_path)


Project root: C:\Users\Mist\Documents\Portfolio\P1. Retail Sales Forecasting and Stock Optimisation
Processed data path: C:\Users\Mist\Documents\Portfolio\P1. Retail Sales Forecasting and Stock Optimisation\data\processed
Evaluation data path: C:\Users\Mist\Documents\Portfolio\P1. Retail Sales Forecasting and Stock Optimisation\data\processed\evaluation


In [7]:
# Load evaluation outputs generated in Notebook 5
eval_path = DATA_PROCESSED / "evaluation"

store_mape = pd.read_csv(eval_path / "store_mape.csv")
store_bias = pd.read_csv(eval_path / "store_bias.csv")

# Sort for clarity
store_mape = store_mape.sort_values("mape", ascending=False)
store_bias = store_bias.sort_values("avg_signed_error", ascending=False)

store_mape.head(10), store_bias.head(10)


(   store      mape
 0     28  6.131339
 1     14  5.713284
 2     40  5.234089
 3     38  4.670553
 4     29  4.560760
 5     35  4.467583
 6     17  4.443776
 7     39  4.425630
 8      7  4.425398
 9     23  4.424411,
    store  avg_signed_error
 0     28      37032.329159
 1     14      21602.673337
 2     20      19678.598350
 3     10      14380.638698
 4     23      14050.890625
 5     27      13994.658957
 6     17      12739.726986
 7      6      11665.222680
 8     25      10012.873441
 9     11       9649.692066)

### Combined Interpretation: Store-Level MAPE and Bias

Stores with the highest forecast error (MAPE) indicate locations where demand is more
volatile, irregular, or influenced by factors not captured by the model. Stores with the
highest positive bias represent locations where the model systematically predicts too high.

Several stores appear in both categories, including Stores 28, 14, 23, and 17. These stores
represent the highest operational risk because the model is both inaccurate and consistently
over optimistic. This combination may lead to persistent overstocking and increased holding
costs.

Other stores, such as 40, 38, and 29, exhibit high error but without major bias, indicating
unpredictable behaviour rather than systematic over forecasting.

A third group, including Stores 20, 10, 27, 6, 25, and 11, shows strong over forecasting bias
despite relatively acceptable accuracy. These stores would benefit from a simple correction
factor to improve level accuracy.

Understanding these differences helps target interventions such as safety stock adjustments,
localised modelling, or bias correction for specific stores, rather than applying a one-
size-fits-all strategy.


# Business Recommendations

Based on the modelling and evaluation findings, the following recommendations are made
to improve forecasting reliability and operational efficiency across the Walmart store
network.

---

## 1. Deploy the model across the majority of stores

The Random Forest model consistently outperforms the baseline, reducing forecast error
by approximately 30 percent. Most stores achieve MAPE between 2 percent and 4 percent,
indicating that the model is sufficiently accurate for operational use.

**Business impact:**
- Reduced stockouts.
- Lower excess inventory.
- More accurate labour and replenishment planning.

---

## 2. Apply additional monitoring to high-variance stores

Stores such as 28, 14, and 40 show higher error and strong positive bias. These stores
likely experience irregular demand patterns, localised events, or data quality issues.

**Recommendations:**
- Apply localised correction factors (e.g. downward adjustment).
- Maintain slightly higher safety stock to mitigate uncertainty.
- Conduct store-level review of anomalies or event-driven patterns.
- Consider adding store-specific features (e.g. promotions, local events).

---

## 3. Investigate systematic over forecasting bias

The model shows positive bias in several stores, meaning it tends to over predict.

**Operational risks:**
- Excess stock and holding costs.
- Increased markdown likelihood.
- Reduced cash flow efficiency.

**Actions:**
- Apply post-model bias correction.
- Re-examine lag and rolling feature windows for volatile stores.
- Consider store clustering to reduce generalisation error.

---

## 4. Consider feature enhancements for future iterations

The model relies heavily on the 4-week rolling mean, indicating that short-term history
captures most of the signal. External features (temperature, CPI, unemployment) contribute
little and may need refinement.

**Potential future enhancements:**
- Promotion/markdown indicators.
- Holiday-specific dummy variables.
- External event data (sports, weather anomalies).
- Store category segmentation.

---

## 5. Integrate predictions into dashboards for real-time use

The saved predictions and store-level metrics can be integrated into Power BI or Tableau.

**This enables:**
- Visual monitoring of forecast performance.
- Dynamic filtering by store, region, or product hierarchy.
- Early detection of stores drifting off pattern.

---

## 6. Schedule regular model retraining

Demand patterns shift over time. Retraining every 8–12 weeks ensures the model stays aligned
with current trends.

---
